In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Data Manipulation
import numpy as np
import pandas as pd
# Visualisation
import matplotlib.pyplot as plt
# Dataset exploring
import os
# Dataset generation
from keras.preprocessing import image_dataset_from_directory
from keras.preprocessing.image import ImageDataGenerator
# Transfert learning
from keras.applications import VGG16
# Optimizer
from keras.optimizers import Adam
# Keras layers
from keras.layers import Input, Dense, Dropout, Flatten, AveragePooling2D
# Keras model
from keras.models import Model

In [ ]:
classes = []
class_counter = 0

for dirname, _, filenames in os.walk('../input/dlai5-bad-apples/train/train/'):
    if dirname.endswith('/'):
        continue
    else:
        classes.append({dirname.split('/')[-1]: 0})
    file_count = 0
    for filename in filenames:
        file_count += 1
    classes[class_counter][dirname.split('/')[-1]] = file_count
    class_counter += 1
    
print('{:<15} {:<15}'.format('Class', 'Number of instances'))
print()
for d in classes:
    [(k, v)] = d.items()
    print('{:<15} {:<15}'.format(k, v))

In [ ]:
counts = []
labels = []
for d in classes:
    [(k, v)] = d.items()
    labels.append(k)
    counts.append(v)

plt.figure()
plt.bar(range(len(counts)), counts, color = ['yellow', 'orange', 'orange', 'green', 'green', 'yellow'], alpha = .7)
plt.xticks(range(len(counts)), labels, rotation = 30)
plt.title('Count of each label in our training data')
plt.show()

In [ ]:
TRAIN_PATH = '../input/dlai5-bad-apples/train/train/'
TEST_PATH = '../input/dlai5-bad-apples/test/test/'

In [ ]:
datagen = ImageDataGenerator(
    rotation_range = 30, 
    zoom_range = .3, 
    horizontal_flip = True, 
    vertical_flip = True, 
    validation_split = .3
)

train_ds = datagen.flow_from_directory(
    directory = TRAIN_PATH,
    target_size = (150, 150),
    color_mode = 'rgb',
    class_mode = 'categorical',
    subset = 'training'
)

validation_ds = datagen.flow_from_directory(
    directory = TRAIN_PATH,
    target_size = (150, 150),
    color_mode = 'rgb',
    class_mode = 'categorical',
    subset = 'validation'
)

In [ ]:
vgg16 = VGG16(include_top = False, weights = 'imagenet', input_shape = (150, 150, 3))
vgg16.trainable = False

In [ ]:
X_input = Input(shape = (150, 150, 3))
X = vgg16(X_input)
X = AveragePooling2D(pool_size = (3, 3), strides = 2, padding = 'valid',name = 'AvgPool2D')(X)
X = Flatten(name = 'Flatten')(X)
X = Dense(200, activation = 'relu', name = 'Dense1')(X)
X = Dropout(.1)(X)
X = Dense(100, activation = 'relu', name = 'Dense2')(X)
X = Dropout(.1)(X)
X = Dense(6, activation = 'softmax', name = 'Dense3')(X)

model = Model(inputs = X_input, outputs = X, name = 'Fruit_Classifer')

model.summary()

In [ ]:
optimizer = Adam(learning_rate = 0.001)

model.compile(optimizer = optimizer, loss = 'categorical_crossentropy', metrics = ['accuracy'])

_ = model.fit(train_ds, validation_data = validation_ds, epochs = 5, batch_size = 32)

In [ ]:
test_ds = image_dataset_from_directory(
    TEST_PATH,
    label_mode = 'categorical',
    color_mode = 'rgb',
    image_size = (150, 150)
)

In [ ]:
results = model.evaluate(test_ds)

print('{:<20} {:<20}'.format('Test loss', 'Test accuracy'))
print('{:<20} {:<20}'.format(np.round(results[0], 2), np.round(results[1], 2)))

In [ ]:
model.save('fruit_preds')

In [ ]:
from keras.preprocessing import image as image_utils
from keras.applications.imagenet_utils import preprocess_input

import matplotlib.image as mpimg

def make_predictions(image_path):
    image = image_utils.load_img(image_path, target_size=(150, 150))
    image = image_utils.img_to_array(image)
    image = image.reshape(1,150,150,3)
    image = preprocess_input(image)
    preds = model.predict(image)
    return preds.argmax()


import os
file = open('submission.csv','w')
file.write(f'ID,Code\n')
for dirname, _, filenames in os.walk('../input/dlai5-bad-apples/validation/'):
    for filename in filenames:
        p = os.path.join(dirname, filename)
        r = make_predictions(p) + 1
        file.write(f'{filename},{r}\n')
file.close()
print('output done.')

In [ ]:
from keras.preprocessing import image as image_utils
from keras.applications.imagenet_utils import preprocess_input
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.image as mpimg
import os
import random

def make_predictions(image_path):
    image = image_utils.load_img(image_path, target_size=(150, 150))
    image = image_utils.img_to_array(image)
    image = image.reshape(1,150,150,3)
    image = preprocess_input(image)
    preds = (model.predict(image)).argmax()
    return preds+1

def get_label(pred):
    if pred==1:
        return 'freshapples'
    if pred==2:
        return 'freshbananas'
    if pred==3:
        return 'freshoranges'
    if pred==4:
        return 'rottenapples'
    if pred==5:
        return 'rottenbananas'
    if pred==6:
        return 'rottenoranges'

base_dir = '../input/dlai5-bad-apples'
train_dir = os.path.join(base_dir,'train/train')
test_dir = os.path.join(base_dir, 'test/test')
valid_dir = os.path.join(base_dir, 'validation')

def show_samples(folder, rows=3, cols=5):
    images = []
    for i in range(rows*cols):
        fruit = random.choice(os.listdir(folder))
        image = random.choice(os.listdir(os.path.join(folder, fruit)))
        images.append([fruit, os.path.join(folder, fruit, image)])
        p = os.path.join(folder, fruit, image)
        print(get_label(make_predictions(p)))
    _, axes = plt.subplots(rows,cols, figsize=(18,5))

    axes = axes.flatten()
    for img, ax in zip(images, axes):
        ax.imshow(mpimg.imread(img[1]))
        ax.set_title(img[0])
        ax.set_axis_off()
        
    plt.show()

show_samples(valid_dir)

In [ ]:
import pandas
df = pandas.read_csv('submission.csv')
print(df.head)